# Phase 9 — Project summary
Final results dashboard. Every number comes from the measured report files via `src/readme_results.py` (the same code that fills the README).

In [1]:
import sys
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt
from IPython.display import Markdown, display
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "base.yaml").is_file())
sys.path.insert(0, str(root))
from src import readme_results as rr
rd = next((p for p in (root / "artifacts" / "reports", root / "results" / "reports") if (p / "test_results.json").is_file()), root / "artifacts" / "reports")
R = rr.load_reports(rd)
print("reports read from:", rd.relative_to(root), "| found:", [k for k in ("audit", "val", "test", "service", "cal", "rag") if R[k]])

reports read from: artifacts/reports | found: ['audit']


## Headline and results (generated)

In [2]:
display(Markdown(rr.headline(R))); display(Markdown(rr.results(R)))

_Headline results are generated from the measured reports; run the pipeline, then `python scripts/generate_readme_results.py`._


### Data

5,183 abstracts; 1,109 claims (queries): 647 train / 162 validation (seeded, query-grouped split of the official train queries) / 300 held-out test.

### Validation (model selection)

_Not generated yet: run `python scripts/evaluate_models.py --stage validation`._

### Held-out test (evaluated once)

_Not generated yet: run `python scripts/evaluate_models.py --stage test --confirm-single-test-evaluation`._

### Training (CPU, 8 GB RAM class)

_No fine-tuning reports found._

### Serving (packaged exact-FAISS service, CPU, sequential single-user)

_Not generated yet: run `python scripts/validate_search_service.py`._

### Optional RAG layer

_Not generated yet: run `python scripts/calibrate_rag_gate.py`._


## Validation vs held-out test (nDCG@10 with 95% bootstrap CI)

In [3]:
if R["val"] and R["test"]:
    names = [k for k in R["test"]["models"] if k in R["val"]["models"]]
    fig, ax = plt.subplots(figsize=(6.5, 3.4)); w = 0.38
    for j, (split, getter) in enumerate((("validation", lambda k: (R["val"]["models"][k]["metrics"]["ndcg@10"], R["val"]["models"][k]["bootstrap_ci"]["ndcg@10"])),
                                         ("held-out test", lambda k: (R["test"]["models"][k]["test_metrics"]["ndcg@10"], R["test"]["models"][k]["test_bootstrap_ci"]["ndcg@10"])))):
        m = [getter(k)[0] for k in names]; lo = [getter(k)[0] - getter(k)[1]["ci_low"] for k in names]; hi = [getter(k)[1]["ci_high"] - getter(k)[0] for k in names]
        ax.bar([i + (j - 0.5) * w for i in range(len(names))], m, w, yerr=[lo, hi], capsize=3, label=split)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=10); ax.set(ylabel="nDCG@10", title="Selected model vs baseline"); ax.legend(); plt.tight_layout(); plt.show()
else:
    print("Validation and/or test results not available yet.")

Validation and/or test results not available yet.


## Reproducibility and publication checks

In [4]:
rows = {}
for name in ("fresh_env_report_lite.json", "fresh_env_report_full.json", "repo_hygiene_report.json", "app_smoke_report.json", "search_service_report.json", "rag_validation_report.json"):
    p = rd / name
    if p.is_file():
        import json
        d = json.loads(p.read_text(encoding="utf-8")); rows[name] = {"status": d.get("status"), "mode": d.get("mode", "")}
display(pd.DataFrame(rows).T if rows else pd.DataFrame({"note": ["no verification reports found yet"]}))
print("Run: python scripts/verify_fresh_env.py | python scripts/check_repo_hygiene.py | python scripts/generate_readme_results.py --check")

,note
0,no verification reports found yet


Run: python scripts/verify_fresh_env.py | python scripts/check_repo_hygiene.py | python scripts/generate_readme_results.py --check


## Caveats
Sparse positive-only judgments; small query sets; one training seed; shared-corpus benchmark; possible overlap between the base model's pretraining mix and the corpus;
similarity is not truth; optional answers are structurally checked, not fact-checked. The dataset is CC BY-NC 2.0: see `docs/DATA_AND_LICENSES.md`.